In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
gender_data = pd.read_csv("/kaggle/input/titanic/gender_submission.csv")
gender_data.head()

In [ ]:
train = pd.read_csv("/kaggle/input/titanic/train.csv")
train.head()

In [ ]:
train.shape
#matches expected output: n = 891

In [ ]:
test = pd.read_csv("/kaggle/input/titanic/test.csv")
test.head()

In [ ]:
test.shape
#matches expected ouput: n = 418

In [ ]:
#Look for null values
test.isna().sum()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
#Looking at gender counts

gender_plt = train['Sex'].value_counts().plot(kind='bar', title= 'Number of males and females aboard the Titanic')
gender_plt.set_xlabel('')

In [ ]:
#Frequency of Pclass

class_plt = train['Pclass'].plot(kind='hist',bins = 3, xticks = range(1,4),
                     title='Frequency of social class groups aboard the Titanic')

class_plt.set_xlabel('P Class')

In [ ]:
# Explore patterns

#Feature engineering

In [ ]:
train.isna().sum() 

In [ ]:
#Observe the various data types
train.dtypes

In [ ]:
#look for duplicates
train.loc[train.duplicated()]

In [ ]:
#Every feature we change for train, we have to do the same for test if those are going to be what we use for randomforest

In [ ]:
train.loc[train['Pclass'] == 1, 'SES'] = 'Upper'
train.loc[train['Pclass'] == 2, 'SES'] = 'Middle'
train.loc[train['Pclass'] == 3, 'SES'] = 'Lower'

test.loc[test['Pclass'] == 1, 'SES'] = 'Upper'
test.loc[test['Pclass'] == 2, 'SES'] = 'Middle'
test.loc[test['Pclass'] == 3, 'SES'] = 'Lower'

In [ ]:
train.loc[train['Survived'] == 0, 'Survival'] = 'Died'
train.loc[train['Survived'] == 1, 'Survival'] = 'Survived'

train.loc[train['Sex'] == 'male', 'Gender'] = 1
train.loc[train['Sex']== 'female', 'Gender'] = 2

train['Gender'] = train['Gender'].astype(int)

## Do the same for test data so that the features appear in both datasets

test.loc[test['Sex'] == 'male', 'Gender'] = 1
test.loc[test['Sex']== 'female', 'Gender'] = 2

test['Gender'] = test['Gender'].astype(int)


In [ ]:
#train.loc[train['Age'].isnull()]
#n=177

In [ ]:
################################################### TRAIN #####################################################

lower_male = train.loc[(train['SES']=='Lower') & (train['Sex']=='male') & (~train['Age'].isnull())]
lower_fem = train.loc[(train['SES']=='Lower') & (train['Sex']=='female') & (~train['Age'].isnull())]
mid_male = train.loc[(train['SES']=='Middle') & (train['Sex']=='male') & (~train['Age'].isnull())]
mid_fem = train.loc[(train['SES']=='Middle') & (train['Sex']=='female') & (~train['Age'].isnull())] 
upper_fem = train.loc[(train['SES']=='Upper') & (train['Sex']=='female') & (~train['Age'].isnull())] 
upper_male = train.loc[(train['SES']=='Upper') & (train['Sex']=='male') & (~train['Age'].isnull())]

avg_age = lambda x: round(x['Age'].mean(), 1)

l_male = avg_age(lower_male)
l_fem = avg_age(lower_fem)
m_male = avg_age(mid_male)
m_fem = avg_age(mid_fem)
u_fem = avg_age(upper_fem)
u_male = avg_age(upper_male)

#################################################### TEST #####################################################

lower_male_test = test.loc[(test['SES']=='Lower') & (test['Sex']=='male') & (~test['Age'].isnull())]
lower_fem_test = test.loc[(test['SES']=='Lower') & (test['Sex']=='female') & (~test['Age'].isnull())]
mid_male_test = test.loc[(test['SES']=='Middle') & (test['Sex']=='male') & (~test['Age'].isnull())]
mid_fem_test = test.loc[(test['SES']=='Middle') & (test['Sex']=='female') & (~test['Age'].isnull())] 
upper_fem_test = test.loc[(test['SES']=='Upper') & (test['Sex']=='female') & (~test['Age'].isnull())] 
upper_male_test = test.loc[(test['SES']=='Upper') & (test['Sex']=='male') & (~test['Age'].isnull())]

l_male_test = avg_age(lower_male_test)
l_fem_test = avg_age(lower_fem_test)
m_male_test = avg_age(mid_male_test)
m_fem_test = avg_age(mid_fem_test)
u_fem_test = avg_age(upper_fem_test)
u_male_test = avg_age(upper_male_test)

In [ ]:
########################################## TRAIN ###################################################

train.loc[(train['SES']=='Lower') & (train['Sex']=='male') & (train['Age'].isnull()), "Age"] = l_male
train.loc[(train['SES']=='Lower') & (train['Sex']=='female') & (train['Age'].isnull()),"Age"] = l_fem
train.loc[(train['SES']=='Middle') & (train['Sex']=='female') & (train['Age'].isnull()),"Age"] = m_fem
train.loc[(train['SES']=='Middle') & (train['Sex']=='male') & (train['Age'].isnull()),"Age"] = m_male
train.loc[(train['SES']=='Upper') & (train['Sex']=='female') & (train['Age'].isnull()),"Age"] = u_fem
train.loc[(train['SES']=='Upper') & (train['Sex']=='male') & (train['Age'].isnull()), "Age"] = u_male

####################################### TEST #########################################################

test.loc[(test['SES']=='Lower') & (test['Sex']=='male') & (test['Age'].isnull()), "Age"] = l_male_test
test.loc[(test['SES']=='Lower') & (test['Sex']=='female') & (test['Age'].isnull()),"Age"] = l_fem_test
test.loc[(test['SES']=='Middle') & (test['Sex']=='female') & (test['Age'].isnull()),"Age"] = m_fem_test
test.loc[(test['SES']=='Middle') & (test['Sex']=='male') & (test['Age'].isnull()),"Age"] = m_male_test
test.loc[(test['SES']=='Upper') & (test['Sex']=='female') & (test['Age'].isnull()),"Age"] = u_fem_test
test.loc[(test['SES']=='Upper') & (test['Sex']=='male') & (test['Age'].isnull()), "Age"] = u_male_test

In [ ]:
test['Age'].isna().sum()

In [ ]:
train['Age'].isna().sum()

In [ ]:
train.loc[train['Age'] <= 5, 'Age_cat'] = 0 # young child
train.loc[(train['Age'] >= 6) & (train['Age'] <15), 'Age_cat'] = 1 # older child
train.loc[(train['Age'] >= 15) & (train['Age'] < 26), 'Age_cat'] = 2 # adult
train.loc[(train['Age'] >= 26) & (train['Age'] < 36), 'Age_cat'] = 3 # older adult
train.loc[train['Age'] > 35, 'Age_cat'] = 4 #elderly-ish ---- life expectancy was a lot lower then than now

https://chnm.gmu.edu/cyh/primary-sources/24.html

In [ ]:
test.loc[test['Age'] <= 5, 'Age_cat'] = 0 # young child
test.loc[(test['Age'] >= 6) & (test['Age'] <15), 'Age_cat'] = 1 # older child
test.loc[(test['Age'] >= 15) & (test['Age'] < 26), 'Age_cat'] = 2 # adult
test.loc[(test['Age'] >= 26) & (test['Age'] < 36), 'Age_cat'] = 3 # older adult
test.loc[test['Age'] > 35, 'Age_cat'] = 4 #elderly-ish ---- life expectancy was a lot lower then than now

In [ ]:
#perhaps there is a correlation b/w departure loc and survival
#lets explore

train['Embarked'].isna().sum() # two values missing

#we can see which group(s) (social class) account for the missing values and take the mode for the dep. loc. for that specific group

train.loc[train['Embarked'].isna()] #interesting: same ticket, fare price, pclass, survived, and cabin

## they also do not identify having parents, children, spouse, or sibling

## nevertheless

emb = train.loc[(train['Sex']=='female') & (train['Pclass']==1) & (train['Age_cat']==4.0)]['Embarked'].mode()
emb = emb[0] #C is the most common embarkment location for all of the descriptive info we have regarding the missing passengers

train.loc[train['Embarked'].isnull(), "Embarked"] = emb
#check train.loc[train['Embarked'].isna()]

In [ ]:
tickets = train.loc[train['Ticket'].duplicated()]['Ticket']

train[train['Ticket'].isin(tickets)].sort_values('Ticket')

#if there are duplicate train tickets, this can be a better indication of whether or not people are traveling alone or with others
#The previous example was a perfect demonstration of how SibSp and Parch can = 0 but have dup tickets

train.loc[~train['Ticket'].duplicated(),"Dup_tickets"] = 0 
train.loc[train['Ticket'].isin(tickets), "Dup_tickets"] = 1

In [ ]:
test_tick = test.loc[test['Ticket'].duplicated()]['Ticket']
test[test['Ticket'].isin(test_tick)].sort_values('Ticket')

test.loc[~test['Ticket'].duplicated(),"Dup_tickets"] = 0 
test.loc[test['Ticket'].isin(test_tick), "Dup_tickets"] = 1

In [ ]:
train['fam_size'] = train['SibSp'] + train['Parch'] + train['Dup_tickets']+ 1
test['fam_size'] = test['SibSp'] + test['Parch'] + test['Dup_tickets'] + 1

In [ ]:
#Taking a look at survival rate for upper class first to see is there is a positive correlation between social class and survival

total_upper = len(train.query('SES == "Upper"'))
upper_survived = train.query('SES == "Upper" and Survival=="Survived"')\
['Survived'].sum()
upper_surv_rt = round((upper_survived/total_upper)*100,2)

upper_surv_rt

In [ ]:
print(f'{upper_surv_rt}% of upper class passengers survived Titanic.')

In [ ]:
#Using what we did for the upper class passengers, we can create a fn to do the same for the other classes

def class_surv(df, class_):
    total = df.loc[df['SES']==class_]
    total_surv_rt = total.loc[total['Survival']== 'Survived']
    
    return round((len(total_surv_rt)/len(total))*100, 2)

In [ ]:
#Did upper just to make sure the function produced the same ouput as before
class_surv(train, 'Upper')

In [ ]:
low_surv = class_surv(train, 'Lower')
low_surv

In [ ]:
mid_surv = class_surv(train, 'Middle')
mid_surv

In [ ]:
train['SES'].value_counts()

In [ ]:
print(f'There appears to be a correlation between social class and survival rates. Although lower class passengers were the most populous of the three, they had the lowest survival rate ({low_surv}%) showing that less than a quarter of them survived. Additionally, nearly half of middle class passengers ({mid_surv})% survived, while {upper_surv_rt}% of upper class passengers survived.')

In [ ]:
lower_fare = test.loc[(test['Sex']== 'male') & (test['SES']=='Lower') & (~test['Fare'].isnull())]

l_fare = round(lower_fare['Fare'].mean(),2)

test.loc[(test['SES']=='Lower') & (test['Sex']=='male') & (test['Fare'].isnull()), "Fare"] = l_fare

In [ ]:
test['Fare'].isna().sum()

In [ ]:
train['Fare'].isna().sum()

In [ ]:
#any correlation between survival and parent-child (Parch)

x = train.query('Survived == 0')
y = train.query('Survived == 1')

x['SibSp'].value_counts() #majority of the people who died did not have siblings or a spouse
y['SibSp'].value_counts()

x['Parch'].value_counts() #majority of the people who died did not have parents or children
y['Parch'].value_counts()

In [ ]:
train

In [ ]:
from sklearn.ensemble import RandomForestClassifier as rfc

y = train['Survived']

features = ['Sex', 'Pclass', 'Fare', 'Age_cat','fam_size', 'Embarked','SibSp', 'Dup_tickets']

In [ ]:
X = pd.get_dummies(train[features])
X_test = pd.get_dummies(test[features])

model = rfc(n_estimators=100, max_depth=5, random_state=1)
model.fit(X,y)

In [ ]:
prediction = model.predict(X_test)

pred_out = pd.DataFrame({'PassengerId': test.PassengerId, 'Survived': prediction})
pred_out.to_csv("submission.csv", index=False)
print("Your submission was successfully saved!")